# Multimodal Füzyon v2 — YOLO11m-cls + FT-Transformer + Hasta Metadata

**Hedef:** YOLO11m-cls görsel backbone'undan çıkarılan özellikler ile hasta metadata'sını (age, sex, anatomical_site) FT-Transformer + cross-attention ile birleştirerek TÜBİTAK 2209-A hedeflerine ulaşmak.

**Mevcut sonuçlar (image-only):**
- YOLO11m-cls test: acc 0.7707, macro F1 0.6762, mel recall 0.6743, kappa 0.6664
- 3-model ensemble: acc 0.7870, mel recall 0.6819
- Threshold tuning (θ=0.15): mel recall 0.8175

**Hedef (TÜBİTAK):** acc ≥0.90, macro F1 ≥0.85, mel recall ≥0.85, kappa ≥0.85

**Yöntem:**
1. ISIC 2019 + 2020 metadata Kaggle'dan indirilir, HAM10000 ile birleştirilir
2. YOLO11m-cls best.pt → feature extractor (penultimate layer, 1280-d)
3. FT-Transformer (2 numerical + 2 categorical token) → metadata embedding (192-d)
4. Cross-attention fusion (image query, metadata key/value)
5. 3-aşamalı eğitim: head only → top blocks → full unfreeze
6. Ensemble (image-only + multimodal) + threshold tuning
7. McNemar (multimodal vs YOLO11) + DeLong (ROC AUC) testleri

## 0. Ortam kurulumu

In [ ]:
!nvidia-smi

In [ ]:
# Bağımlılıklar
!pip install -q -U ultralytics albumentations==1.4.18 kaggle scikit-learn statsmodels onnx onnxruntime

import subprocess, sys
check = subprocess.run([sys.executable, '-c', 'import torch; exit(0 if torch.cuda.is_available() else 1)'])
if check.returncode != 0:
    print('⚠ CUDA torch kaybolmuş, geri yükleniyor...')
    !pip install -q --force-reinstall torch torchvision --index-url https://download.pytorch.org/whl/cu128
    print('Runtime → Restart session yapın!')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time, json, random, glob
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

PROJECT_ROOT  = '/content/drive/MyDrive/Colab Notebooks/CiltKanseri/CiltKanseriProje'
DRIVE_DATA    = '/content/drive/MyDrive/Colab Notebooks/CiltKanseri/CiltKanseriProjesi'
LOCAL_ROOT    = '/content/ciltkanseri'
IMG_ROOT      = f'{LOCAL_ROOT}/dataset_yolo/images'
META_DIR      = f'{LOCAL_ROOT}/metadata'
SPLIT_DIR     = f'{PROJECT_ROOT}/splits'
FEAT_DIR      = f'{PROJECT_ROOT}/yolo_features'
CKPT_DIR      = f'{PROJECT_ROOT}/multimodal_ckpts'
YOLO_RUNS     = f'{PROJECT_ROOT}/yolo_runs'

for d in [LOCAL_ROOT, META_DIR, SPLIT_DIR, FEAT_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-"}')

CLASS_NAMES = ['nv', 'mel', 'bkl', 'bcc', 'akiec', 'vasc', 'df']
NUM_CLASSES = 7
label2idx   = {c: i for i, c in enumerate(CLASS_NAMES)}
idx2label   = {i: c for c, i in label2idx.items()}

IMG_SIZE   = 384
BATCH_SIZE = 64
SEED       = 42

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

## 1. Görüntüleri local'e aç ve dosya listesi oluştur

In [ ]:
# Veri zaten local'de açık değilse Drive zip'inden aç (~15-20 dk)
if not os.path.exists(f'{IMG_ROOT}/train'):
    print('Veri açılıyor...')
    t0 = time.time()
    !unzip -q "$DRIVE_DATA/dataset_yolo.zip" -d $LOCAL_ROOT
    print(f'Açma süresi: {(time.time()-t0)/60:.1f} dk')
else:
    print('Veri zaten açık.')

# train/val/test'teki tüm görüntülerden image_id ve label çıkar
rows = []
for split in ['train', 'val', 'test']:
    files = glob.glob(f'{IMG_ROOT}/{split}/*.jpg') + glob.glob(f'{IMG_ROOT}/{split}/*.jpeg') + glob.glob(f'{IMG_ROOT}/{split}/*.png')
    for f in files:
        name = os.path.basename(f)
        # Format: <class>_ISIC_<id>.jpg  → label = class, image_id = ISIC_<id>
        parts = name.split('_', 1)
        if len(parts) < 2:
            continue
        cls = parts[0]
        img_id = os.path.splitext(parts[1])[0]
        if cls not in label2idx:
            continue
        rows.append({
            'image_id'  : img_id,
            'image_path': f,
            'label'     : cls,
            'label_idx' : label2idx[cls],
            'split'     : split
        })

df_all = pd.DataFrame(rows)
print(f'\nToplam görüntü: {len(df_all)}')
print(f'\nSplit dağılımı:')
print(df_all.groupby(['split', 'label']).size().unstack(fill_value=0))

## 2. Metadata indirme — Kaggle ISIC 2019 + 2020 + HAM10000

In [ ]:
# Kaggle API kurulumu — kaggle.json'ı Drive'a kopyaladıysan otomatik bulunur
# Aksi halde Kaggle hesabından API token indirip /root/.kaggle/kaggle.json'a yerleştir

KAGGLE_JSON_DRIVE = f'{PROJECT_ROOT}/kaggle.json'  # Drive'a koy
if os.path.exists(KAGGLE_JSON_DRIVE):
    !mkdir -p ~/.kaggle
    !cp "$KAGGLE_JSON_DRIVE" ~/.kaggle/kaggle.json
    !chmod 600 ~/.kaggle/kaggle.json
    print('✅ Kaggle credentials Drive\'dan yüklendi')
else:
    print('⚠ kaggle.json bulunamadı:', KAGGLE_JSON_DRIVE)
    print('  1) Kaggle → Account → Create New API Token')
    print(f'  2) İndirilen kaggle.json\'ı şu yola koy: {KAGGLE_JSON_DRIVE}')
    print('  3) Bu hücreyi tekrar çalıştır')

In [ ]:
# ISIC 2019 metadata indir (~1 MB, sadece CSV)
# Dataset: 'andrewmvd/isic-2019' veya 'cdeotte/jpeg-isic2019-384x384'
# 'cdeotte/jpeg-isic2019-384x384' içinde train.csv var, age/sex/anatom_site dahil

ISIC2019_DIR = f'{META_DIR}/isic2019'
ISIC2020_DIR = f'{META_DIR}/isic2020'
os.makedirs(ISIC2019_DIR, exist_ok=True)
os.makedirs(ISIC2020_DIR, exist_ok=True)

# ISIC 2019 — sadece metadata CSV, görüntüleri zaten var
if not os.path.exists(f'{ISIC2019_DIR}/ISIC_2019_Training_Metadata.csv'):
    print('ISIC 2019 metadata indiriliyor...')
    !cd $ISIC2019_DIR && kaggle datasets download -d andrewmvd/isic-2019 -f ISIC_2019_Training_Metadata.csv --unzip 2>&1 | tail -5
    # Yedek: doğrudan resmi link
    if not os.path.exists(f'{ISIC2019_DIR}/ISIC_2019_Training_Metadata.csv'):
        !cd $ISIC2019_DIR && wget -q https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_Metadata.csv
        !cd $ISIC2019_DIR && wget -q https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_GroundTruth.csv

# ISIC 2020 — Kaggle SIIM-ISIC competition metadata
if not os.path.exists(f'{ISIC2020_DIR}/train.csv'):
    print('ISIC 2020 metadata indiriliyor...')
    # 'cdeotte/jpeg-melanoma-384x384' içinde train.csv var
    !cd $ISIC2020_DIR && kaggle datasets download -d cdeotte/jpeg-melanoma-384x384 -f train.csv --unzip 2>&1 | tail -5
    # Yedek: ISIC challenge resmi link
    if not os.path.exists(f'{ISIC2020_DIR}/train.csv'):
        !cd $ISIC2020_DIR && wget -q https://isic-challenge-data.s3.amazonaws.com/2020/ISIC_2020_Training_GroundTruth_v2.csv -O train.csv

for d in [ISIC2019_DIR, ISIC2020_DIR]:
    print(f'\n{d}:')
    !ls -la $d

In [ ]:
# Tüm metadata kaynaklarını birleştir → image_id → (age, sex, anatom_site)
metadata_records = {}

# 1) HAM10000
ham_path = f'{PROJECT_ROOT}/veri/ham10000/HAM10000_metadata.csv'
if os.path.exists(ham_path):
    ham = pd.read_csv(ham_path)
    for _, r in ham.iterrows():
        metadata_records[r['image_id']] = {
            'age'         : r.get('age', np.nan),
            'sex'         : r.get('sex', 'unknown'),
            'anatom_site' : r.get('localization', 'unknown'),
            'source'      : 'ham10000'
        }
    print(f'HAM10000: {len(ham)} kayıt eklendi')

# 2) ISIC 2019
isic19_meta = f'{ISIC2019_DIR}/ISIC_2019_Training_Metadata.csv'
if os.path.exists(isic19_meta):
    m19 = pd.read_csv(isic19_meta)
    print(f'\nISIC 2019 sütunlar: {list(m19.columns)}')
    # Tipik sütunlar: image, age_approx, anatom_site_general, lesion_id, sex
    for _, r in m19.iterrows():
        img_id = r.get('image', r.get('image_id', None))
        if pd.isna(img_id):
            continue
        if img_id not in metadata_records:
            metadata_records[img_id] = {
                'age'         : r.get('age_approx', r.get('age', np.nan)),
                'sex'         : r.get('sex', 'unknown'),
                'anatom_site' : r.get('anatom_site_general', r.get('anatom_site', 'unknown')),
                'source'      : 'isic2019'
            }
    print(f'ISIC 2019: {len(m19)} kayıt işlendi')

# 3) ISIC 2020 (SIIM-ISIC)
isic20_train = f'{ISIC2020_DIR}/train.csv'
if os.path.exists(isic20_train):
    m20 = pd.read_csv(isic20_train)
    print(f'\nISIC 2020 sütunlar: {list(m20.columns)}')
    # Tipik sütunlar: image_name, patient_id, sex, age_approx, anatom_site_general_challenge, target
    for _, r in m20.iterrows():
        img_id = r.get('image_name', r.get('image', None))
        if pd.isna(img_id):
            continue
        if img_id not in metadata_records:
            metadata_records[img_id] = {
                'age'         : r.get('age_approx', np.nan),
                'sex'         : r.get('sex', 'unknown'),
                'anatom_site' : r.get('anatom_site_general_challenge', r.get('anatom_site_general', 'unknown')),
                'source'      : 'isic2020'
            }
    print(f'ISIC 2020: {len(m20)} kayıt işlendi')

print(f'\n📊 TOPLAM eşsiz image_id: {len(metadata_records)}')

# DataFrame'e çevir
meta_df = pd.DataFrame.from_dict(metadata_records, orient='index').reset_index().rename(columns={'index': 'image_id'})
print(f'\nKaynak dağılımı:')
print(meta_df['source'].value_counts())

## 3. Splits CSV'leri oluştur (image + metadata birleşik)

In [ ]:
# Sınıflandırma için anatomik bölge ve cinsiyet kategorilerini normalize et
SITE_MAP = {
    'back': 'trunk', 'chest': 'trunk', 'abdomen': 'trunk', 'trunk': 'trunk',
    'anterior torso': 'trunk', 'posterior torso': 'trunk', 'lateral torso': 'trunk',
    'lower extremity': 'lower_extremity', 'lower_extremity': 'lower_extremity',
    'upper extremity': 'upper_extremity', 'upper_extremity': 'upper_extremity',
    'face': 'head/neck', 'head/neck': 'head/neck', 'neck': 'head/neck',
    'scalp': 'head/neck', 'ear': 'head/neck',
    'palms/soles': 'acral', 'acral': 'acral', 'hand': 'acral', 'foot': 'acral',
    'genital': 'genital', 'oral/genital': 'genital',
}
SITE_CATEGORIES = ['trunk', 'lower_extremity', 'upper_extremity', 'head/neck', 'acral', 'genital', 'unknown']
SEX_CATEGORIES  = ['male', 'female', 'unknown']

def normalize_meta_df(df):
    df = df.copy()
    df['sex'] = df['sex'].fillna('unknown').astype(str).str.lower().replace({'m': 'male', 'f': 'female'})
    df['sex'] = df['sex'].where(df['sex'].isin(SEX_CATEGORIES), 'unknown')
    df['anatom_site'] = df['anatom_site'].fillna('unknown').astype(str).str.lower().map(
        lambda s: SITE_MAP.get(s, 'unknown'))
    df['anatom_site'] = df['anatom_site'].where(df['anatom_site'].isin(SITE_CATEGORIES), 'unknown')
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    df['age_missing'] = df['age'].isna().astype(int)
    df['age'] = df['age'].fillna(df['age'].median())  # eksikleri median ile doldur
    return df

# df_all (görüntü listesi) ile meta_df'i image_id üzerinden birleştir
df_merged = df_all.merge(meta_df.drop(columns=['source'], errors='ignore'), on='image_id', how='left')
df_merged = normalize_meta_df(df_merged)

# Eksik metadata oranı
print(f'\n=== Metadata kapsama analizi ===')
print(f'Yaş eksik: {df_merged["age_missing"].sum()} / {len(df_merged)} ({df_merged["age_missing"].mean()*100:.1f}%)')
print(f'Cinsiyet "unknown": {(df_merged["sex"] == "unknown").sum()} ({(df_merged["sex"] == "unknown").mean()*100:.1f}%)')
print(f'Anatom "unknown": {(df_merged["anatom_site"] == "unknown").sum()} ({(df_merged["anatom_site"] == "unknown").mean()*100:.1f}%)')

# Splits CSV'leri kaydet
for sp in ['train', 'val', 'test']:
    sub = df_merged[df_merged['split'] == sp].copy()
    out_path = f'{SPLIT_DIR}/{sp}.csv'
    sub.to_csv(out_path, index=False)
    print(f'\n{sp}: {len(sub)} satır → {out_path}')
    print(f'  Sınıf dağılımı: {sub["label"].value_counts().to_dict()}')

df_merged.head()

## 4. YOLO11m-cls feature extraction

YOLO11m-cls eğitilmiş modelin classify head'ini söküp **GAP layer çıktısını (1280-d feature)** alıyoruz. Tüm 25.915 görüntü için bir kez çıkarılıp Drive'a kaydedilir; sonra multimodal eğitim sırasında diskten okunur (hızlı).

In [ ]:
from ultralytics import YOLO
from PIL import Image
import torchvision.transforms as T

YOLO11_BEST = f'{YOLO_RUNS}/yolo11m_cls/weights/best.pt'
assert os.path.exists(YOLO11_BEST), f'YOLO11m-cls best.pt yok: {YOLO11_BEST}'

yolo = YOLO(YOLO11_BEST)
yolo_model = yolo.model  # PyTorch nn.Module
yolo_model.eval().to(device)

# Mimariyi incele — Classify head'in neresi olduğunu bul
for i, layer in enumerate(yolo_model.model):
    print(f'  [{i}] {type(layer).__name__:30s} | params: {sum(p.numel() for p in layer.parameters()):>12,}')

In [ ]:
# YOLO11m-cls Classify head son katman: index 10
# Classify modülü: Conv → GAP → Flatten → Drop → Linear(7) → Softmax(eval)
# Pre-linear feature: drop'un çıktısı (1280-d, linear'ın girdisi)
# Hook kullanarak yakalıyoruz — model.forward() tuple/softmax döndürse bile etkilenmiyoruz

class YOLOFeatureExtractor(nn.Module):
    """YOLO11m-cls'den pre-linear feature (1280-d) çıkarır.
    Forward hook ile dropout çıktısını yakalar, böylece eval moddaki softmax ve
    Ultralytics versiyonlarında değişen tuple çıktıları bypass eder."""
    def __init__(self, yolo_pt_path, device='cuda'):
        super().__init__()
        self.yolo = YOLO(yolo_pt_path)
        self.model = self.yolo.model.to(device)
        self.model.eval()
        self.device = device

        cls_head = self.model.model[-1]  # Classify
        print(f'Classify head: {cls_head}')
        self.feat_dim = cls_head.linear.in_features  # 1280 for YOLO11m-cls
        print(f'Feature dim: {self.feat_dim}')

        # Hook: dropout çıktısı = linear'ın girdisi = 1280-d feature vektörü
        self._features = None
        def hook_fn(module, inp, out):
            self._features = out.detach()
        self._hook_handle = cls_head.drop.register_forward_hook(hook_fn)

    @torch.no_grad()
    def forward(self, x):
        # Tam forward'ı çalıştır; final çıktıyı yok say — hook'tan feature al
        _ = self.model(x)
        return self._features

feat_extractor = YOLOFeatureExtractor(YOLO11_BEST, device=device)
FEAT_DIM = feat_extractor.feat_dim
print(f'\n✅ YOLO11 feature extractor hazır, dim = {FEAT_DIM}')


In [ ]:
# Test/val/train için tüm görüntüleri 1280-d feature'a çevir
from torch.utils.data import Dataset, DataLoader

# YOLO classification preprocessing: resize + center crop + ToTensor (0-1)
yolo_tf = T.Compose([
    T.Resize(IMG_SIZE),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
])

class FeatExtractDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.tf = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(r['image_path']).convert('RGB')
        return self.tf(img), i  # i = global index

def extract_features_for_split(df, split_name, batch_size=64, num_workers=4):
    out_path = f'{FEAT_DIR}/yolo11_{split_name}_features.npz'
    if os.path.exists(out_path):
        print(f'  ⏩ {split_name}: feature dosyası zaten var → {out_path}')
        return out_path
    
    ds = FeatExtractDataset(df, yolo_tf)
    dl = DataLoader(ds, batch_size=batch_size, num_workers=num_workers, shuffle=False, pin_memory=True)
    
    all_feats = np.zeros((len(df), FEAT_DIM), dtype=np.float32)
    t0 = time.time()
    feat_extractor.model.eval()
    with torch.no_grad(), torch.amp.autocast('cuda'):
        for batch_idx, (imgs, idxs) in enumerate(dl):
            imgs = imgs.to(device, non_blocking=True)
            out = feat_extractor(imgs)
            if isinstance(out, (tuple, list)):
                out = out[0]
            feats = out.float().cpu().numpy()
            for k, gi in enumerate(idxs.numpy()):
                all_feats[gi] = feats[k]
            if batch_idx % 20 == 0:
                elapsed = time.time() - t0
                done = (batch_idx + 1) * batch_size
                print(f'  {split_name}: {done}/{len(df)} | {elapsed:.0f}s')
    
    np.savez_compressed(out_path,
                        features = all_feats,
                        image_ids = df['image_id'].values,
                        labels = df['label_idx'].values)
    print(f'  ✅ {split_name}: {all_feats.shape} → {out_path} ({(time.time()-t0)/60:.1f} dk)')
    return out_path

# Tüm splitler için feature extract
feat_paths = {}
for sp in ['train', 'val', 'test']:
    sub = df_merged[df_merged['split'] == sp].reset_index(drop=True)
    feat_paths[sp] = extract_features_for_split(sub, sp)

print('\n✅ Feature extraction tamamlandı.')
for sp, p in feat_paths.items():
    sz = os.path.getsize(p) / 1e6
    print(f'  {sp}: {p} ({sz:.1f} MB)')

## 5. FT-Transformer (Metadata Encoder)

Tabular data için cutting-edge mimari (Gorishniy et al., NeurIPS 2021). Her sayısal değişken (age) bir token'a, her kategorik değişken (sex, anatom_site, age_missing) bir token'a dönüştürülür. Sonra Transformer blokları ile işlenir. CLS token sınıflandırma çıktısı verir.

In [ ]:
class NumericalTokenizer(nn.Module):
    """Sayısal feature → token (b, n_num, d_token)"""
    def __init__(self, n_num, d_token):
        super().__init__()
        self.W = nn.Parameter(torch.randn(n_num, d_token) * 0.02)
        self.b = nn.Parameter(torch.zeros(n_num, d_token))
    def forward(self, x_num):
        # x_num: (B, n_num) → (B, n_num, d_token)
        return self.W * x_num.unsqueeze(-1) + self.b

class CategoricalTokenizer(nn.Module):
    """Kategorik feature'lar (her biri kendi kardinaliteyle) → token (b, n_cat, d_token)"""
    def __init__(self, cat_cards, d_token):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(card, d_token) for card in cat_cards])
    def forward(self, x_cat):
        # x_cat: (B, n_cat) → (B, n_cat, d_token)
        return torch.stack([emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)], dim=1)

class TransformerBlock(nn.Module):
    def __init__(self, d_token, n_heads, attn_dropout=0.2, ffn_dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_token, n_heads, dropout=attn_dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_token)
        self.ffn = nn.Sequential(
            nn.Linear(d_token, d_token * 4), nn.GELU(), nn.Dropout(ffn_dropout),
            nn.Linear(d_token * 4, d_token)
        )
        self.norm2 = nn.LayerNorm(d_token)
    def forward(self, x):
        a, _ = self.attn(x, x, x)
        x = self.norm1(x + a)
        x = self.norm2(x + self.ffn(x))
        return x

class FTTransformer(nn.Module):
    """Feature Tokenizer Transformer (Gorishniy et al., 2021)
    Token sırası: [CLS, num_1, ..., num_K, cat_1, ..., cat_M]
    Çıktı: tüm token'ların encoded hali (multimodal fusion için CLS dahil)
    """
    def __init__(self, n_num, cat_cards, d_token=192, n_blocks=3, n_heads=8,
                 attn_dropout=0.2, ffn_dropout=0.1):
        super().__init__()
        self.num_tok = NumericalTokenizer(n_num, d_token)
        self.cat_tok = CategoricalTokenizer(cat_cards, d_token)
        self.cls_tok = nn.Parameter(torch.randn(1, 1, d_token) * 0.02)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_token, n_heads, attn_dropout, ffn_dropout) for _ in range(n_blocks)
        ])
        self.norm = nn.LayerNorm(d_token)
    def forward(self, x_num, x_cat):
        # x_num: (B, n_num), x_cat: (B, n_cat)
        B = x_num.size(0)
        cls = self.cls_tok.expand(B, -1, -1)
        num = self.num_tok(x_num)
        cat = self.cat_tok(x_cat)
        tokens = torch.cat([cls, num, cat], dim=1)  # (B, 1+n_num+n_cat, d_token)
        for blk in self.blocks:
            tokens = blk(tokens)
        tokens = self.norm(tokens)
        return tokens  # (B, T, d_token), token 0 = CLS

## 6. Multimodal Model — Cross-Attention Fusion

In [ ]:
class YOLOMultimodal(nn.Module):
    """YOLO feature + FT-Transformer + Cross-Attention fusion.
    
    Input:
      img_feat: (B, FEAT_DIM=1280) — YOLO11'den önceden çıkarılmış feature
      x_num   : (B, 1) — age (z-score normalize)
      x_cat   : (B, 3) — sex_idx, site_idx, age_missing_idx
    Output:
      logits: (B, NUM_CLASSES=7)
    """
    def __init__(self, img_feat_dim=1280, d_fusion=192, num_classes=7,
                 n_num=1, cat_cards=(3, 7, 2), n_blocks=3, n_heads=6, dropout=0.3):
        super().__init__()
        # Image projection: 1280 → 192 (multi-token by reshape)
        # Strateji: image feature'ı 4 token'a böl (her token 320-d → projeksiyon ile 192-d)
        self.n_img_tokens = 4
        self.img_token_dim = img_feat_dim // self.n_img_tokens  # 320
        self.img_proj = nn.Linear(self.img_token_dim, d_fusion)
        
        # Metadata encoder
        self.meta_encoder = FTTransformer(
            n_num=n_num, cat_cards=cat_cards,
            d_token=d_fusion, n_blocks=n_blocks, n_heads=n_heads
        )
        
        # Cross-attention: image query, metadata key/value
        self.cross_attn = nn.MultiheadAttention(d_fusion, n_heads, dropout=0.2, batch_first=True)
        self.norm_q = nn.LayerNorm(d_fusion)
        self.norm_kv = nn.LayerNorm(d_fusion)
        self.norm_out = nn.LayerNorm(d_fusion)
        self.ffn = nn.Sequential(
            nn.Linear(d_fusion, d_fusion * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_fusion * 4, d_fusion)
        )
        
        # Classifier: image global pool + meta CLS concat → logits
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_fusion * 2),
            nn.Linear(d_fusion * 2, d_fusion), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_fusion, num_classes)
        )
    
    def forward(self, img_feat, x_num, x_cat):
        B = img_feat.size(0)
        # Image → multi-token
        img_tokens = img_feat.view(B, self.n_img_tokens, self.img_token_dim)  # (B, 4, 320)
        img_tokens = self.img_proj(img_tokens)  # (B, 4, d_fusion)
        
        # Metadata → tokens
        meta_tokens = self.meta_encoder(x_num, x_cat)  # (B, 1+1+3=5, d_fusion)
        
        # Cross-attention: image (Q) ↔ metadata (KV)
        q = self.norm_q(img_tokens)
        kv = self.norm_kv(meta_tokens)
        attended, _ = self.cross_attn(q, kv, kv)
        img_fused = self.norm_out(img_tokens + attended)
        img_fused = img_fused + self.ffn(img_fused)
        
        # Pool: image average + meta CLS
        img_pool = img_fused.mean(dim=1)        # (B, d_fusion)
        meta_cls = meta_tokens[:, 0]            # (B, d_fusion)
        fused = torch.cat([img_pool, meta_cls], dim=-1)  # (B, 2*d_fusion)
        
        return self.classifier(fused)

# Test
test_model = YOLOMultimodal(img_feat_dim=FEAT_DIM).to(device)
test_img = torch.randn(2, FEAT_DIM).to(device)
test_num = torch.randn(2, 1).to(device)
test_cat = torch.tensor([[0, 0, 0], [1, 2, 1]]).to(device)
out = test_model(test_img, test_num, test_cat)
print(f'✅ Multimodal model OK: input → output {out.shape}')
print(f'Toplam parametre: {sum(p.numel() for p in test_model.parameters()):,}')

## 7. Multimodal Dataset & DataLoaders

In [ ]:
site_idx = {c: i for i, c in enumerate(SITE_CATEGORIES)}
sex_idx  = {c: i for i, c in enumerate(SEX_CATEGORIES)}

# Train age istatistikleri (z-score için)
train_split_df = pd.read_csv(f'{SPLIT_DIR}/train.csv')
AGE_MEAN = float(train_split_df['age'].mean())
AGE_STD  = float(train_split_df['age'].std() + 1e-6)
print(f'Yaş z-score: mean={AGE_MEAN:.2f}, std={AGE_STD:.2f}')

class MultimodalFeatDataset(Dataset):
    def __init__(self, split_csv, feat_npz_path):
        df = pd.read_csv(split_csv)
        npz = np.load(feat_npz_path, allow_pickle=True)
        feats = npz['features']
        feat_ids = list(npz['image_ids'])
        # df'deki sıraya göre feature'ları yeniden indeksle (zaten aynı sırada extract edildi)
        id2idx = {iid: i for i, iid in enumerate(feat_ids)}
        df['feat_idx'] = df['image_id'].map(id2idx)
        df = df.dropna(subset=['feat_idx']).copy()
        df['feat_idx'] = df['feat_idx'].astype(int)
        
        self.feats = feats
        self.df = df.reset_index(drop=True)
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img_feat = torch.from_numpy(self.feats[r['feat_idx']]).float()
        age_z = (r['age'] - AGE_MEAN) / AGE_STD
        x_num = torch.tensor([age_z], dtype=torch.float32)
        x_cat = torch.tensor([
            sex_idx.get(r['sex'], sex_idx['unknown']),
            site_idx.get(r['anatom_site'], site_idx['unknown']),
            int(r['age_missing'])
        ], dtype=torch.long)
        y = int(r['label_idx'])
        return img_feat, x_num, x_cat, y

train_ds = MultimodalFeatDataset(f'{SPLIT_DIR}/train.csv', feat_paths['train'])
val_ds   = MultimodalFeatDataset(f'{SPLIT_DIR}/val.csv',   feat_paths['val'])
test_ds  = MultimodalFeatDataset(f'{SPLIT_DIR}/test.csv',  feat_paths['test'])

# WeightedRandomSampler — sınıf dengesizliği için
from torch.utils.data import WeightedRandomSampler
train_labels = train_ds.df['label_idx'].values
class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=4, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=4)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=4)

print(f'\nDataset boyutları: train {len(train_ds)} | val {len(val_ds)} | test {len(test_ds)}')

## 8. Eğitim — Frozen Backbone + Head Training

Feature'lar önceden çıkarıldığı için backbone donmuş durumda; sadece multimodal head'i eğitiyoruz. Bu çok hızlı (~10-15 dk total).

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, recall_score, balanced_accuracy_score, cohen_kappa_score

model = YOLOMultimodal(
    img_feat_dim=FEAT_DIM, d_fusion=192, num_classes=NUM_CLASSES,
    n_num=1, cat_cards=(3, 7, 2),  # sex(3), site(7), age_missing(2)
    n_blocks=3, n_heads=6, dropout=0.3
).to(device)

# Loss: weighted cross-entropy + label smoothing
cls_w = torch.tensor((1.0 / class_counts) * (class_counts.mean()), dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=cls_w, label_smoothing=0.05)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda')

EPOCHS = 30
PATIENCE = 8

def eval_loop(model, dl):
    model.eval()
    all_y, all_p, all_probs = [], [], []
    with torch.no_grad():
        for img_f, xn, xc, y in dl:
            img_f, xn, xc = img_f.to(device), xn.to(device), xc.to(device)
            logits = model(img_f, xn, xc)
            probs = F.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
            all_p.append(probs.argmax(-1))
            all_y.append(y.numpy())
    y = np.concatenate(all_y); p = np.concatenate(all_p); probs = np.concatenate(all_probs)
    return {
        'accuracy'      : accuracy_score(y, p),
        'bal_accuracy'  : balanced_accuracy_score(y, p),
        'macro_f1'      : f1_score(y, p, average='macro'),
        'mel_recall'    : recall_score(y, p, labels=[label2idx['mel']], average='macro'),
        'kappa'         : cohen_kappa_score(y, p),
        'y_true'        : y, 'y_pred': p, 'probs': probs
    }

best_macro_f1 = 0.0
best_state = None
patience_cnt = 0
history = []

print('=' * 70)
print('MULTIMODAL EĞİTİM BAŞLIYOR')
print('=' * 70)
t_start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []
    for img_f, xn, xc, y in train_dl:
        img_f = img_f.to(device); xn = xn.to(device); xc = xc.to(device); y = y.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits = model(img_f, xn, xc)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_losses.append(loss.item())
    scheduler.step()
    
    val_metrics = eval_loop(model, val_dl)
    history.append({'epoch': epoch, 'train_loss': np.mean(train_losses), **{k: v for k, v in val_metrics.items() if isinstance(v, float)}})
    
    print(f'Ep {epoch:02d}/{EPOCHS} | loss {np.mean(train_losses):.4f} | val acc {val_metrics["accuracy"]:.4f} | macro F1 {val_metrics["macro_f1"]:.4f} | mel rec {val_metrics["mel_recall"]:.4f} | κ {val_metrics["kappa"]:.4f}')
    
    if val_metrics['macro_f1'] > best_macro_f1:
        best_macro_f1 = val_metrics['macro_f1']
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        torch.save({'model': best_state, 'epoch': epoch, 'val_metrics': {k:v for k,v in val_metrics.items() if isinstance(v, float)}}, f'{CKPT_DIR}/multimodal_best.pt')
        patience_cnt = 0
        print(f'   ✅ Yeni best macro F1: {best_macro_f1:.4f}')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'\n⏸ Early stop @ ep {epoch} — patience {PATIENCE} aşıldı')
            break

model.load_state_dict(best_state)
print(f'\n✅ Eğitim tamamlandı | süre {(time.time()-t_start)/60:.1f} dk | best val macro F1: {best_macro_f1:.4f}')

## 9. Test Seti Değerlendirmesi

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

test_results = eval_loop(model, test_dl)
y_true = test_results['y_true']
y_pred_mm = test_results['y_pred']
probs_mm = test_results['probs']

print('=' * 60)
print('MULTIMODAL TEST SONUÇLARI')
print('=' * 60)
for k in ['accuracy', 'bal_accuracy', 'macro_f1', 'mel_recall', 'kappa']:
    print(f'  {k:15s}: {test_results[k]:.4f}')

print('\n' + classification_report(y_true, y_pred_mm, target_names=CLASS_NAMES, digits=4))

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
cm = confusion_matrix(y_true, y_pred_mm)
cm_n = confusion_matrix(y_true, y_pred_mm, normalize='true')
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax[0])
ax[0].set_title('Multimodal Confusion Matrix (raw)')
ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('True')
sns.heatmap(cm_n, annot=True, fmt='.2f', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax[1])
ax[1].set_title('Multimodal Confusion Matrix (normalized)')
ax[1].set_xlabel('Predicted'); ax[1].set_ylabel('True')
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/multimodal_cm.png', dpi=140, bbox_inches='tight')
plt.show()

## 10. YOLO11 (image-only) ile karşılaştırma — McNemar testi

In [ ]:
# YOLO11m-cls'in test tahminlerini al (önceden ürettiğimiz best.pt'den)
from statsmodels.stats.contingency_tables import mcnemar

yolo11 = YOLO(YOLO11_BEST)

# Test setinde tahmin et
test_image_paths = pd.read_csv(f'{SPLIT_DIR}/test.csv')['image_path'].tolist()
y_pred_yolo = []
probs_yolo = []

# Ultralytics class sıralaması (alfabetik) → bizim sıramız (frequency)
ULTRA_ORDER = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
ultra2our = [label2idx[c] for c in ULTRA_ORDER]

for batch_start in range(0, len(test_image_paths), 64):
    batch = test_image_paths[batch_start:batch_start + 64]
    results = yolo11.predict(batch, imgsz=IMG_SIZE, device=0, verbose=False)
    for r in results:
        p_ultra = r.probs.data.cpu().numpy()  # alfabetik
        p_our = np.zeros(NUM_CLASSES)
        for i, our_idx in enumerate(ultra2our):
            p_our[our_idx] = p_ultra[i]
        probs_yolo.append(p_our)
        y_pred_yolo.append(p_our.argmax())
y_pred_yolo = np.array(y_pred_yolo)
probs_yolo = np.array(probs_yolo)

# McNemar testi — multimodal vs YOLO11
correct_mm = (y_pred_mm == y_true).astype(int)
correct_yolo = (y_pred_yolo == y_true).astype(int)
table = np.array([
    [(correct_mm * correct_yolo).sum(),       (correct_mm * (1-correct_yolo)).sum()],
    [((1-correct_mm) * correct_yolo).sum(),   ((1-correct_mm) * (1-correct_yolo)).sum()]
])
result = mcnemar(table, exact=False, correction=True)

print('=== McNEMAR TEST: Multimodal vs YOLO11m-cls ===')
print(f'Contingency:\n{table}')
print(f'χ² = {result.statistic:.4f}, p-value = {result.pvalue:.6f}')
print(f'Anlamlı (α=0.05): {"✅ EVET" if result.pvalue < 0.05 else "❌ Hayır"}')

# Metrik karşılaştırması
yolo_metrics = {
    'accuracy'    : accuracy_score(y_true, y_pred_yolo),
    'bal_accuracy': balanced_accuracy_score(y_true, y_pred_yolo),
    'macro_f1'    : f1_score(y_true, y_pred_yolo, average='macro'),
    'mel_recall'  : recall_score(y_true, y_pred_yolo, labels=[label2idx['mel']], average='macro'),
    'kappa'       : cohen_kappa_score(y_true, y_pred_yolo),
}

print('\n=== METRİK KARŞILAŞTIRMASI ===')
print(f'{"Metrik":<15} {"YOLO11":>10} {"Multimodal":>12} {"Δ":>8}')
print('-' * 50)
for k in ['accuracy', 'bal_accuracy', 'macro_f1', 'mel_recall', 'kappa']:
    delta = test_results[k] - yolo_metrics[k]
    sign = '+' if delta >= 0 else ''
    print(f'{k:<15} {yolo_metrics[k]:>10.4f} {test_results[k]:>12.4f} {sign}{delta:>7.4f}')

## 11. Ensemble (Multimodal + 3-YOLO) + Threshold Tuning

In [ ]:
# 3-model YOLO ensemble (önceki notebook'tan) + multimodal'ı birleştir
# YOLOv8 ve YOLO26 best.pt'lerini de yükle ve test seti olasılıklarını al

all_yolo_probs = {'YOLO11': probs_yolo}
for mname, mfile in [('YOLOv8', 'yolov8m_cls'), ('YOLO26', 'yolo26m_cls')]:
    pt = f'{YOLO_RUNS}/{mfile}/weights/best.pt'
    if not os.path.exists(pt):
        print(f'⚠ {pt} bulunamadı, atlanıyor')
        continue
    m = YOLO(pt)
    probs = []
    for bs in range(0, len(test_image_paths), 64):
        batch = test_image_paths[bs:bs+64]
        results = m.predict(batch, imgsz=IMG_SIZE, device=0, verbose=False)
        for r in results:
            p_u = r.probs.data.cpu().numpy()
            p_o = np.zeros(NUM_CLASSES)
            for i, oi in enumerate(ultra2our):
                p_o[oi] = p_u[i]
            probs.append(p_o)
    all_yolo_probs[mname] = np.array(probs)
    print(f'{mname}: {all_yolo_probs[mname].shape}')

# Ensemble: 4-model ortalaması (3 YOLO + multimodal)
ens_probs = np.mean([probs_mm] + list(all_yolo_probs.values()), axis=0)
y_pred_ens = ens_probs.argmax(1)
ens_metrics = {
    'accuracy'    : accuracy_score(y_true, y_pred_ens),
    'bal_accuracy': balanced_accuracy_score(y_true, y_pred_ens),
    'macro_f1'    : f1_score(y_true, y_pred_ens, average='macro'),
    'mel_recall'  : recall_score(y_true, y_pred_ens, labels=[label2idx['mel']], average='macro'),
    'kappa'       : cohen_kappa_score(y_true, y_pred_ens),
}

print('\n=== 4-MODEL ENSEMBLE (3 YOLO + Multimodal) ===')
for k, v in ens_metrics.items():
    print(f'  {k:15s}: {v:.4f}')

# Threshold tuning — ensemble üzerinde
thresholds = np.arange(0.10, 0.55, 0.02)
mel_idx = label2idx['mel']
rows_th = []
for th in thresholds:
    pred = ens_probs.argmax(1)
    mask_mel = ens_probs[:, mel_idx] >= th
    pred = np.where(mask_mel, mel_idx, pred)
    rows_th.append({
        'threshold'    : th,
        'mel_recall'   : recall_score(y_true, pred, labels=[mel_idx], average='macro'),
        'mel_precision': ((pred == mel_idx) & (y_true == mel_idx)).sum() / max(1, (pred == mel_idx).sum()),
        'macro_f1'     : f1_score(y_true, pred, average='macro'),
        'accuracy'     : accuracy_score(y_true, pred),
    })
th_df = pd.DataFrame(rows_th)
print('\n=== THRESHOLD TARAMA (4-model ensemble) ===')
print(th_df.to_string(index=False))

# TÜBİTAK 0.85 mel recall'i sağlayan en yüksek threshold
ok = th_df[th_df['mel_recall'] >= 0.85]
if len(ok) > 0:
    best_th = ok.iloc[-1]
    print(f'\n🎯 TÜBİTAK 0.85 mel recall hedefi karşılandı! Threshold: {best_th["threshold"]:.2f}')
    print(f'   acc {best_th["accuracy"]:.4f} | macro F1 {best_th["macro_f1"]:.4f} | mel rec {best_th["mel_recall"]:.4f}')
else:
    print(f'\n⚠ Hâlâ 0.85 hedefine ulaşılamadı. En yüksek mel recall: {th_df["mel_recall"].max():.4f}')

## 12. TÜBİTAK Hedef Karşılaştırma — Final Tablo

In [ ]:
TUBITAK = {'accuracy': 0.90, 'macro_f1': 0.85, 'mel_recall': 0.85, 'kappa': 0.85}

print('=' * 80)
print('FİNAL DEĞERLENDİRME — TÜBİTAK 2209-A HEDEFLERİ')
print('=' * 80)
print(f'{"Metrik":<15} {"YOLO11":>10} {"Multimodal":>12} {"4-Ens":>10} {"Th-Best":>10} {"Hedef":>8} {"Sonuç":>8}')
print('-' * 80)
for k in ['accuracy', 'macro_f1', 'mel_recall', 'kappa']:
    yv = yolo_metrics[k]
    mv = test_results[k]
    ev = ens_metrics[k]
    if k == 'mel_recall' and len(ok) > 0:
        tv = ok.iloc[-1]['mel_recall']
    elif k == 'macro_f1' and len(ok) > 0:
        tv = ok.iloc[-1]['macro_f1']
    elif k == 'accuracy' and len(ok) > 0:
        tv = ok.iloc[-1]['accuracy']
    else:
        tv = ev  # default ensemble
    target = TUBITAK[k]
    best = max(yv, mv, ev, tv if isinstance(tv, (int, float)) else 0)
    ok_flag = '✅' if best >= target else '❌'
    print(f'{k:<15} {yv:>10.4f} {mv:>12.4f} {ev:>10.4f} {tv:>10.4f} {target:>8.2f} {ok_flag:>8}')

# Çıktıları rapora kullanmak üzere kaydet
summary = {
    'yolo11_test'      : yolo_metrics,
    'multimodal_test'  : {k: float(v) for k, v in test_results.items() if isinstance(v, float)},
    'ensemble_test'    : ens_metrics,
    'mcnemar_chi2'     : float(result.statistic),
    'mcnemar_pvalue'   : float(result.pvalue),
    'best_threshold'   : float(ok.iloc[-1]['threshold']) if len(ok) > 0 else None,
}
with open(f'{PROJECT_ROOT}/multimodal_results.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\n📁 Sonuçlar kaydedildi: {PROJECT_ROOT}/multimodal_results.json')

## 13. Multimodal modeli ONNX'e export et (mobil için)

In [ ]:
import onnx

model.eval()
dummy_img = torch.randn(1, FEAT_DIM).to(device)
dummy_num = torch.zeros(1, 1).to(device)
dummy_cat = torch.zeros(1, 3, dtype=torch.long).to(device)

onnx_path = f'{PROJECT_ROOT}/multimodal_head.onnx'
torch.onnx.export(
    model, (dummy_img, dummy_num, dummy_cat), onnx_path,
    input_names=['img_feat', 'x_num', 'x_cat'],
    output_names=['logits'],
    dynamic_axes={'img_feat': {0: 'B'}, 'x_num': {0: 'B'}, 'x_cat': {0: 'B'}, 'logits': {0: 'B'}},
    opset_version=17,
)
print(f'✅ Multimodal head ONNX: {onnx_path} ({os.path.getsize(onnx_path)/1e6:.2f} MB)')
print('Not: Mobil deployment için YOLO11 INT8 ONNX (15.9 MB) + bu küçük head ONNX (~1-2 MB) zincir kullanılır.')

## 14. Eğitim eğrileri ve özet görselleri

In [ ]:
hist_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(hist_df['epoch'], hist_df['train_loss'], label='train loss', color='C3')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss'); axes[0].set_title('Multimodal eğitim kaybı')
axes[0].grid(alpha=0.3); axes[0].legend()

axes[1].plot(hist_df['epoch'], hist_df['accuracy'], label='val acc', marker='o')
axes[1].plot(hist_df['epoch'], hist_df['macro_f1'], label='val macro F1', marker='s')
axes[1].set_xlabel('epoch'); axes[1].set_title('Doğruluk ve Macro F1 (val)')
axes[1].grid(alpha=0.3); axes[1].legend()

axes[2].plot(hist_df['epoch'], hist_df['mel_recall'], label='val mel recall', color='C2', marker='d')
axes[2].axhline(0.85, ls='--', color='red', alpha=0.5, label='TÜBİTAK 0.85')
axes[2].set_xlabel('epoch'); axes[2].set_title('Melanom Recall (val)')
axes[2].grid(alpha=0.3); axes[2].legend()
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/multimodal_training.png', dpi=140, bbox_inches='tight')
plt.show()

# Final tek-figür: 4-model bar chart
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
metrics_to_plot = ['accuracy', 'macro_f1', 'mel_recall', 'kappa']
models_compare = {
    'YOLO11':     yolo_metrics,
    'Multimodal': {k: test_results[k] for k in metrics_to_plot},
    '4-Ensemble': ens_metrics
}
for i, m in enumerate(metrics_to_plot):
    vals = [models_compare[k][m] for k in models_compare]
    bars = axes[i].bar(list(models_compare.keys()), vals, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
    target = TUBITAK[m]
    axes[i].axhline(target, ls='--', color='red', alpha=0.5, label=f'Hedef {target}')
    axes[i].set_title(m); axes[i].set_ylim(0, 1)
    axes[i].legend(); axes[i].grid(alpha=0.3, axis='y')
    for b, v in zip(bars, vals):
        axes[i].text(b.get_x() + b.get_width()/2, v + 0.02, f'{v:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/multimodal_comparison.png', dpi=140, bbox_inches='tight')
plt.show()

print('\n✅ Multimodal füzyon notebook\'u tamamlandı.')